In [4]:
import requests
import pandas as pd

def get_closed_markets(limit=100, offset=0):
    url = "https://gamma-api.polymarket.com/markets"
    
    params = {
        "closed": "true",
        "limit": limit,
        "offset": offset
    }
    
    r = requests.get(url, params=params)
    r.raise_for_status()
    
    data = r.json()
    
    # normalize into dataframe
    df = pd.json_normalize(data)
    
    return df

# Example usage
df = get_closed_markets(limit=50)
print(df[["question", "outcomes", "resolution", "closed"]].head())

KeyError: "['resolution'] not in index"

In [ ]:
def get_all_closed_markets(max_markets=1000):
    all_data = []
    offset = 0
    limit = 100
    
    while offset < max_markets:
        df = get_closed_markets(limit=limit, offset=offset)
        
        if df.empty:
            break
        
        all_data.append(df)
        offset += limit
    
    return pd.concat(all_data, ignore_index=True)

df_all = get_all_closed_markets(500)
print(len(df_all))

In [ ]:
import requests
import pandas as pd

BASE = "https://clob.polymarket.com"

def get_some_markets():
    r = requests.get(f"{BASE}/simplified-markets", timeout=20)
    r.raise_for_status()
    return r.json()["data"]

def get_price_history(asset_id, start_ts=None, end_ts=None, interval="1d", fidelity=60):
    params = {
        "market": asset_id,   # must be a real token_id / asset id
        "interval": interval,
        "fidelity": fidelity,
    }
    if start_ts is not None:
        params["startTs"] = start_ts
    if end_ts is not None:
        params["endTs"] = end_ts

    r = requests.get(f"{BASE}/prices-history", params=params, timeout=20)
    print("Request URL:", r.url)
    print("Status:", r.status_code)
    print("Body:", r.text[:500])  # useful if something still fails
    r.raise_for_status()

    hist = r.json()["history"]
    df = pd.DataFrame(hist)

    if df.empty:
        return df

    df["datetime"] = pd.to_datetime(df["t"], unit="s", utc=True)
    df = df.rename(columns={"t": "timestamp", "p": "price"})
    return df[["timestamp", "datetime", "price"]]

# 1) discover a real token_id
markets = get_some_markets()

# pick the first market and its first token
first_market = markets[0]
first_token = first_market["tokens"][0]

print("Outcome:", first_token["outcome"])
print("Token ID:", first_token["token_id"])


df = get_price_history(
    asset_id=first_token["token_id"],
    interval="1d",
    fidelity=60
)

print(df.head())

Outcome: Arizona State
Token ID: 73470541315377973562501025254719659796416871135081220986683321361000395461644
Request URL: https://clob.polymarket.com/prices-history?market=73470541315377973562501025254719659796416871135081220986683321361000395461644&interval=1d&fidelity=60
Status: 200
Body: {"history":[]}

Empty DataFrame
Columns: []
Index: []
